# Exercise 2 — Common Dictionary Keys and Paired Values

## Objective

Given two dictionaries, create a new dictionary containing only the keys that occur in **both** dictionaries.

For each common key, the result should contain a tuple with:

1. the value from the first dictionary, and
2. the value from the second dictionary.

For example:

```python
d1 = {'a': 1, 'b': 2, 'c': 3, 'd': 4}
d2 = {'b': 20, 'c': 30, 'y': 40, 'z': 50}
```

should produce:

```python
{'b': (2, 20), 'c': (3, 30)}
```

The solution should be Pythonic and should not build the result by repeatedly assigning items to an initially empty dictionary.

## Example data

In [1]:
d1 = {'a': 1, 'b': 2, 'c': 3, 'd': 4}
d2 = {'b': 20, 'c': 30, 'y': 40, 'z': 50}

## Core Pythonic solution

Dictionary keys behave like sets for many set operations.

The intersection operator `&` can therefore be used to find keys that exist in both dictionaries:

```python
d1.keys() & d2.keys()
```

Once the common keys are known, a dictionary comprehension can construct the result directly.

In [2]:
def common_items(d1, d2):
    """Return common keys with values paired as (d1_value, d2_value)."""
    return {
        key: (d1[key], d2[key])
        for key in d1.keys() & d2.keys()
    }

In [3]:
result = common_items(d1, d2)
print(result)

{'b': (2, 20), 'c': (3, 30)}


The order of keys is not part of the exercise requirements, so either ordering of `b` and `c` is valid.

Conceptually, the result is:

```python
{'b': (2, 20), 'c': (3, 30)}
```

---

## Production-quality implementation

For reusable code, we can improve the function in several ways:

- accept any `Mapping`, not only concrete `dict` objects;
- preserve a predictable ordering;
- use generic type hints;
- document behavior explicitly;
- avoid modifying either input.

### Why preserve the first mapping's order?

A set intersection is ideal for identifying common keys, but sets do not represent an application-level ordering contract.

For reproducible output, the implementation below iterates through the first mapping and keeps only keys also present in the second mapping. This preserves the first mapping's iteration order while remaining concise and Pythonic.

In [5]:
from collections.abc import Hashable, Mapping
from typing import TypeVar


K = TypeVar('K', bound=Hashable)
V1 = TypeVar('V1')
V2 = TypeVar('V2')


def common_key_values(
    first: Mapping[K, V1],
    second: Mapping[K, V2],
) -> dict[K, tuple[V1, V2]]:
    """Return values from two mappings for keys present in both.

    Parameters
    ----------
    first:
        The first mapping.
    second:
        The second mapping.

    Returns
    -------
    dict
        A new dictionary whose keys occur in both mappings.
        Each value is a tuple of the form::

            (first[key], second[key])

        Common keys retain their relative iteration order from
        ``first``.

    Notes
    -----
    Neither input mapping is modified.
    """
    return {
        key: (value, second[key])
        for key, value in first.items()
        if key in second
    }

## Run the enhanced implementation

In [6]:
result = common_key_values(d1, d2)
print(result)

{'b': (2, 20), 'c': (3, 30)}


Expected output:

```text
{'b': (2, 20), 'c': (3, 30)}
```

---

## Step-by-step explanation

The key part of the function is:

```python
{
    key: (value, second[key])
    for key, value in first.items()
    if key in second
}
```

For each `(key, value)` pair in `first`:

1. `if key in second` checks whether the same key exists in the second mapping.
2. If it does, the new value becomes `(value, second[key])`.
3. If it does not, the key is omitted.

For the example input:

- `'a'` exists only in `d1` → excluded
- `'b'` exists in both → `'b': (2, 20)`
- `'c'` exists in both → `'c': (3, 30)`
- `'d'` exists only in `d1` → excluded

## Set-intersection approach

The exercise hint specifically points toward set intersection, so here is the corresponding concise implementation:

In [7]:
def common_key_values_set(first, second):
    common_keys = first.keys() & second.keys()

    return {
        key: (first[key], second[key])
        for key in common_keys
    }


print(common_key_values_set(d1, d2))

{'b': (2, 20), 'c': (3, 30)}


Both versions are valid.

The set-intersection implementation mirrors the mathematical definition of the problem very clearly. The order-preserving implementation is useful when deterministic presentation order matters.

---

## Verification tests

In [8]:
expected = {
    'b': (2, 20),
    'c': (3, 30),
}

result = common_key_values(d1, d2)

assert result == expected
assert list(result) == ['b', 'c']

print('Basic test passed.')

Basic test passed.


## Verify that inputs are not modified

In [9]:
first_before = d1.copy()
second_before = d2.copy()

_ = common_key_values(d1, d2)

assert d1 == first_before
assert d2 == second_before

print('Input mappings were not modified.')

Input mappings were not modified.


---

## Edge-case tests

Good reusable functions should work naturally for empty mappings, mappings with no overlap, complete overlap, different value types, and `None` values.

In [10]:
# Both mappings empty
assert common_key_values({}, {}) == {}

# First mapping empty
assert common_key_values({}, {'a': 1}) == {}

# Second mapping empty
assert common_key_values({'a': 1}, {}) == {}

# No common keys
assert common_key_values(
    {'a': 1, 'b': 2},
    {'x': 10, 'y': 20},
) == {}

# Every key is common
assert common_key_values(
    {'a': 1, 'b': 2},
    {'a': 10, 'b': 20},
) == {
    'a': (1, 10),
    'b': (2, 20),
}

# The values do not need to have the same type
assert common_key_values(
    {'id': 42, 'active': True},
    {'id': 'user-42', 'active': 'yes'},
) == {
    'id': (42, 'user-42'),
    'active': (True, 'yes'),
}

# None is a valid dictionary value
assert common_key_values(
    {'a': None, 'b': 2},
    {'a': 100, 'c': 3},
) == {
    'a': (None, 100),
}

print('All edge-case tests passed.')

All edge-case tests passed.


---

# Added Value: Generalize to Any Number of Dictionaries

The original problem involves exactly two dictionaries. A useful extension is to support **any number of mappings**.

For example:

```python
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'a': 10, 'b': 20, 'x': 30}
d3 = {'a': 100, 'b': 200, 'z': 300}
```

The keys common to all three are `a` and `b`, so the result can be:

```python
{
    'a': (1, 10, 100),
    'b': (2, 20, 200),
}
```

In [11]:
def common_values(*mappings: Mapping) -> dict:
    """Return values for keys that occur in every supplied mapping.

    The returned dictionary preserves the iteration order of the
    first mapping. Each result value is a tuple containing the value
    from every mapping in argument order.

    Parameters
    ----------
    *mappings:
        Two or more mapping objects.

    Returns
    -------
    dict
        Common keys mapped to tuples of corresponding values.

    Raises
    ------
    ValueError
        If fewer than two mappings are supplied.
    """
    if len(mappings) < 2:
        raise ValueError('At least two mappings are required.')

    first, *rest = mappings

    return {
        key: tuple(mapping[key] for mapping in mappings)
        for key in first
        if all(key in mapping for mapping in rest)
    }

## Example with three dictionaries

In [12]:
a = {'python': 10, 'java': 3, 'go': 5}
b = {'python': 20, 'java': 7, 'rust': 8}
c = {'python': 30, 'java': 9, 'c++': 4}

result = common_values(a, b, c)
print(result)

{'python': (10, 20, 30), 'java': (3, 7, 9)}


Expected output:

```text
{'python': (10, 20, 30), 'java': (3, 7, 9)}
```

## Test the generalized function

In [13]:
assert common_values(a, b, c) == {
    'python': (10, 20, 30),
    'java': (3, 7, 9),
}

assert common_values(
    {'a': 1},
    {'b': 2},
    {'c': 3},
) == {}

try:
    common_values({'a': 1})
except ValueError as exc:
    assert str(exc) == 'At least two mappings are required.'
else:
    raise AssertionError('Expected ValueError was not raised.')

print('Generalized-function tests passed.')

Generalized-function tests passed.


---

# Added Value: Explicit Set-Based N-Way Intersection

When only the mathematical set of common keys is required, Python can intersect key views directly.

For multiple mappings, one approach is to progressively intersect their key sets:

In [14]:
def common_values_via_sets(*mappings):
    """Generalized solution using explicit key-set intersection."""
    if len(mappings) < 2:
        raise ValueError('At least two mappings are required.')

    common_keys = set(mappings[0])

    for mapping in mappings[1:]:
        common_keys.intersection_update(mapping)

    return {
        key: tuple(mapping[key] for mapping in mappings)
        for key in common_keys
    }


print(common_values_via_sets(a, b, c))

{'java': (3, 7, 9), 'python': (10, 20, 30)}


This version expresses the set theory especially clearly. The previous `common_values()` implementation is preferable when retaining the first mapping's order is useful.

---

## Why not use `.get()` here?

A tempting implementation might use:

```python
second.get(key)
```

However, once we have already established that a key exists in `second`, direct indexing is preferable:

```python
second[key]
```

Using `.get()` could also make missing keys indistinguishable from keys whose legitimate value is `None` if existence were not checked separately.

For this problem, membership testing plus direct indexing communicates the intent precisely.

---

## Complexity analysis

Let:

- **n** = number of items in the first mapping
- **m** = number of items in the second mapping
- **k** = number of keys common to both mappings

For normal Python dictionaries, membership lookup such as:

```python
key in second
```

is **O(1) average-case**.

The order-preserving implementation examines each key of the first mapping once, so its expected running time is:

**Time:** `O(n)` average-case

The result contains `k` entries, therefore:

**Output space:** `O(k)`

The explicit set-intersection solution also requires temporary storage for the intersection, so it can require additional `O(k)` space beyond the returned dictionary.

## A small optimization consideration

When the dictionaries differ greatly in size and output ordering does **not** matter, iterating over an intersection can avoid scanning an unnecessarily large mapping.

When stable, predictable output order is desirable, iterating through the first mapping is often a better API behavior.

The best implementation therefore depends not only on asymptotic complexity, but also on the ordering contract required by the application.

---

# Final Solution

For the original exercise, the most direct solution based on the provided hint is:

```python
def common_items(d1, d2):
    return {
        key: (d1[key], d2[key])
        for key in d1.keys() & d2.keys()
    }
```

For reusable code where predictable ordering is desirable, the recommended version is:

```python
def common_key_values(first, second):
    return {
        key: (value, second[key])
        for key, value in first.items()
        if key in second
    }
```

Both solutions are concise, Pythonic, non-mutating, and avoid incrementally constructing an empty dictionary.